In [1]:
import matplotlib as mpl
import matplotlib.pyplot as plt
%matplotlib inline
import numpy as np
import sklearn
import pandas as pd
import os
import sys
import time
from tqdm.auto import tqdm
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import datasets, transforms
import matplotlib.pyplot as plt

plt.tight_layout()
plt.show()

device = torch.device("cuda:0") if torch.cuda.is_available() else torch.device("cpu")
print(device)



data_dir = './archive'

transform = transforms.Compose([transforms.ToTensor(),
                                transforms.Resize((128, 128)),
                                transforms.Normalize(mean=[0.4363, 0.4328, 0.3291], std=[0.2427, 0.2382, 0.2413])
                                ])

train_dataset = datasets.ImageFolder(root = os.path.join(data_dir, 'training'), transform=transform)
test_dataset = datasets.ImageFolder(root = os.path.join(data_dir, 'validation'), transform=transform)

class_names = train_dataset.classes
print(class_names)

<Figure size 640x480 with 0 Axes>

cuda:0
['n0', 'n1', 'n2', 'n3', 'n4', 'n5', 'n6', 'n7', 'n8', 'n9']


In [2]:
from torch.utils.data import DataLoader

batch_size = 32

train_loader = DataLoader(train_dataset,batch_size=batch_size,shuffle=True,num_workers=4)
test_loader = DataLoader(test_dataset,batch_size=batch_size,shuffle=False,num_workers=4)

In [3]:
import torch.nn as nn

class MonkeyNet(nn.Module):
    def __init__(self,num_classes=10):
        super(MonkeyNet,self).__init__()
        
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 32, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),
            nn.Dropout(0.2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 64, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),
            nn.Dropout(0.3),

            nn.Conv2d(64, 128, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, 128, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),
            nn.Dropout(0.4),

            nn.Conv2d(128, 256, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.Conv2d(256, 256, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2,2),
            nn.Dropout(0.5)
        )
        self.gap = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(128, num_classes),
        )
        
    def forward(self,x):
        x = self.features(x)
        x = self.gap(x)
        x = self.classifier(x)
        return x

model = MonkeyNet(num_classes=20)


In [4]:
import torch

dummy_input = torch.randn(32, 3, 128, 128)
output = model(dummy_input)
print(output.shape)

total_params = 0
for name, param in model.named_parameters():
    if param.requires_grad:
        num_params = param.numel()
        total_params += num_params
        print(name,':',num_params)
print(f"Total Parameters: {total_params}")


torch.Size([32, 20])
features.0.weight : 864
features.1.weight : 32
features.1.bias : 32
features.3.weight : 9216
features.4.weight : 32
features.4.bias : 32
features.8.weight : 18432
features.9.weight : 64
features.9.bias : 64
features.11.weight : 36864
features.12.weight : 64
features.12.bias : 64
features.16.weight : 73728
features.17.weight : 128
features.17.bias : 128
features.19.weight : 147456
features.20.weight : 128
features.20.bias : 128
features.24.weight : 294912
features.25.weight : 256
features.25.bias : 256
features.27.weight : 589824
features.28.weight : 256
features.28.bias : 256
classifier.1.weight : 32768
classifier.1.bias : 128
classifier.4.weight : 2560
classifier.4.bias : 20
Total Parameters: 1208692


In [5]:
import torch.nn as nn
import torch.optim as optim
import my_trainer as mt

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

trainer = mt.Trainer(
    model=model,
    train_loader=train_loader,
    val_loader=test_loader,
    criterion=criterion,
    optimizer=optimizer,
    device=device,
    eval_step=100
)

num_epochs = 20
trainer.train(num_epochs)

Epoch [1/20]  Train Loss: 2.2964  Train Acc: 0.2088
Epoch [2/20]  Train Loss: 1.8672  Train Acc: 0.3099
[Step 100] Val Loss: 1.6246 Val Acc: 0.4265
Epoch [3/20]  Train Loss: 1.8408  Train Acc: 0.3163
Epoch [4/20]  Train Loss: 1.7104  Train Acc: 0.3528
Epoch [5/20]  Train Loss: 1.6332  Train Acc: 0.4066
[Step 200] Val Loss: 1.5861 Val Acc: 0.4007
Epoch [6/20]  Train Loss: 1.6436  Train Acc: 0.3947
Epoch [7/20]  Train Loss: 1.6246  Train Acc: 0.4020
Epoch [8/20]  Train Loss: 1.5480  Train Acc: 0.4202
[Step 300] Val Loss: 1.4694 Val Acc: 0.4449
Epoch [9/20]  Train Loss: 1.4926  Train Acc: 0.4321
Epoch [10/20]  Train Loss: 1.4729  Train Acc: 0.4448
Epoch [11/20]  Train Loss: 1.4092  Train Acc: 0.4704
[Step 400] Val Loss: 1.4259 Val Acc: 0.5037
Epoch [12/20]  Train Loss: 1.3858  Train Acc: 0.4795
Epoch [13/20]  Train Loss: 1.3966  Train Acc: 0.4768
Epoch [14/20]  Train Loss: 1.3616  Train Acc: 0.4777
[Step 500] Val Loss: 1.3660 Val Acc: 0.4779
Epoch [15/20]  Train Loss: 1.2669  Train Acc: 0